# 05 — Attach metadata and find useful associations

Attach metadata to each model at the level its outputs naturally represent.

TEMPTED has one score per subject, so it uses subject-level metadata. MEFISTO has one score per sample, so it can also use time-varying metadata.

The continuous metadata used here are selected explicitly from the fixed DIABIMMUNE metadata rather than discovered automatically. Associations use Spearman correlation and are used to identify informative plots in notebook 06.


In [ ]:
from datetime import datetime
from pathlib import Path

import pandas as pd
from scipy.stats import spearmanr

root = Path(".") if Path("data").exists() else Path("..")
prep = sorted((root / "data" / "preprocessing").iterdir())[-1]
t_dir = sorted((root / "data" / "tempted").iterdir())[-1]
m_dir = sorted((root / "data" / "mefisto").iterdir())[-1]
output = root / "data" / "metadata_analysis" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

# load model outputs and full metadata
metadata = pd.read_csv(prep / "metadata.csv", dtype={"sample_id": str, "subject_id": str})
tempted = pd.read_csv(t_dir / "subject_scores.csv", dtype={"subject_id": str})
mefisto = pd.read_csv(m_dir / "sample_factors.csv", dtype={"sample_id": str, "subject_id": str})

t_dims = [c for c in tempted if c.startswith("component_")]
m_dims = [c for c in mefisto if c.startswith("factor_")]

# continuous subject-level metadata
subject_variables = [
    "gest_time",
    "bf_length",
    "num_abx_treatments",
    "num_abx_first_year",
    "num_aabs",
    "totalige_log",
]

# additional time-varying metadata for mefisto
sample_variables = [
    "age",
    "num_preceeding_abx",
]


In [ ]:
# attach subject-level metadata to tempted

subject_metadata = (
    metadata[
        ["subject_id", "country"]
        + subject_variables
    ]
    .drop_duplicates("subject_id")
)

tempted = tempted.merge(
    subject_metadata,
    on="subject_id",
    how="left",
)

# attach full sample-level metadata to mefisto

mefisto = mefisto.merge(
    metadata,
    on=["sample_id", "subject_id", "age"],
    how="left",
)

# save metadata-rich model tables

tempted.to_csv(
    output / "tempted_with_metadata.csv",
    index=False,
)

mefisto.to_csv(
    output / "mefisto_with_metadata.csv",
    index=False,
)

In [ ]:
# calculate continuous metadata associations

rows = []

for method, frame, dims, variables in [
    (
        "TEMPTED",
        tempted,
        t_dims,
        subject_variables,
    ),
    (
        "MEFISTO",
        mefisto,
        m_dims,
        subject_variables + sample_variables,
    ),
]:
    for variable in variables:
        values = pd.to_numeric(
            frame[variable],
            errors="coerce",
        )

        for dim in dims:
            keep = values.notna() & frame[dim].notna()

            rho = spearmanr(
                values[keep],
                frame.loc[keep, dim],
            ).statistic

            rows.append([
                method,
                variable,
                dim,
                rho,
                abs(rho),
            ])

associations = pd.DataFrame(
    rows,
    columns=[
        "method",
        "metadata",
        "dimension",
        "association",
        "absolute_association",
    ],
)

# save associations
associations.to_csv(
    output / "metadata_associations.csv",
    index=False,
)

print("Saved:", output)

associations.sort_values(
    "absolute_association",
    ascending=False,
)
